# Random Survival Forest (RSF) – Training Pipeline (Versione Aggiornata)

Questo notebook descrive la pipeline di addestramento per il modello **Random Survival Forest** applicato al dataset Head & Neck Cancer.
Rispetto alla versione precedente, sono state introdotte diverse migliorie: **imputazione MICE**, **feature engineering**, **selezione automatica delle feature** tramite permutation importance e una **griglia di iperparametri più mirata** per contrastare l’overfitting, dato il numero contenuto di campioni.

---

### Fase 1: Caricamento e Preprocessing

Il dataset viene caricato e i placeholder `-1` (utilizzati per codificare i valori mancanti nelle variabili categoriali e TNM) vengono convertiti in `NaN`.
Vengono escluse le colonne non numeriche (identificativi, variabili stringa) e le colonne target (`follow_ups.days_to_follow_up` e `demographic.vital_status_DEAD`).
L’obiettivo è ottenere una matrice di sole feature numeriche pronta per le successive fasi di imputazione e modellazione.

---

### Fase 2: Suddivisione Train / Test

Il dataset viene suddiviso in training (75%) e test (25%) utilizzando **stratificazione sull’evento** (`vital_status_DEAD`).
Questo garantisce che la proporzione di pazienti deceduti e censurati sia simile nei due insiemi, condizione necessaria per una valutazione non distorta delle performance.
Il seed `RANDOM_STATE = 25` assicura la riproducibilità dello split.

---

### Fase 3: Imputazione MICE e Feature Engineering

#### Imputazione MICE
Viene applicato l’algoritmo **MICE** (Multiple Imputation by Chained Equations) tramite `IterativeImputer` di scikit‑learn.
La scelta è motivata dalla presenza di pattern di missingness complessi in un dataset clinico; MICE modella ciascuna variabile con missing in funzione delle altre, preservando le relazioni multivariate.
L’imputer viene addestrato **esclusivamente sul training set** (`fit_transform`) e successivamente applicato al test set (`transform`) per evitare qualsiasi forma di *data leakage*.
Il numero di iterazioni è fissato a `max_iter=15`, sufficiente per la convergenza in dataset di dimensioni moderate.

#### Feature Engineering
Sul dataset imputato vengono create nuove feature derivate da combinazioni note in letteratura oncologica:

- **`TNM_Burden`** = `Final_T_enc` + `Final_N_enc`  (sommatoria dei due componenti TNM, rappresenta un indice di carico tumorale).
- **`TNM_Interaction`** = `Final_T_enc` * `Final_N_enc` (interazione moltiplicativa, cattura effetti sinergici).
- **`Total_Treatments`** = somma delle variabili binarie `Trt_PHARMACEUTICAL`, `Trt_RADIATION`, `Trt_SURGERY` (numero di trattamenti ricevuti).
- **`Age_TNM_Risk`** = (`demographic.age_at_index` / 10) * `TNM_Burden` (interazione età‑carico tumorale, normalizzata per mantenere la scala confrontabile).

Queste feature arricchiscono il set informativo e possono aiutare il modello a cogliere relazioni non lineari implicite.

---

### Fase 4: Feature Selection Automatica (sul Training Set)

Per ridurre la dimensionalità ed eliminare feature poco informative, si esegue una selezione basata sull’**importanza mediante permutazione** calcolata su un modello Random Survival Forest preliminare.

- Viene addestrata una Random Survival Forest con parametri conservativi (`n_estimators=100`, `min_samples_leaf=20`) su tutti i dati di training ingegnerizzati.
- Si calcola la permutation importance (5 ripetizioni) utilizzando come metrica il C‑index.
- Vengono selezionate le **prime 8 feature** con importanza maggiore (o tutte se il numero totale è inferiore a 8).

Questo approccio permette di mantenere solo i predittori più rilevanti, riducendo il rumore e il rischio di overfitting, e velocizzando la successiva ottimizzazione degli iperparametri.
La selezione è effettuata **solo sul training set** e poi applicata al test set, garantendo l’assenza di leakage.

---

### Fase 5: Ottimizzazione degli Iperparametri (RSF)

La Random Survival Forest viene ottimizzata mediante **RandomizedSearchCV** con 40 combinazioni casuali e validazione incrociata a 5 fold.
La griglia di parametri è stata progettata per favorire modelli **regolarizzati** e adatti a dataset di dimensioni contenute:

- `n_estimators`: 300, 500 – un numero sufficiente di alberi per stabilizzare le predizioni, senza eccedere in tempi di calcolo.
- `max_depth`: 3, 4, 5 – alberi volutamente poco profondi per limitare la complessità e prevenire l’overfitting.
- `min_samples_split`: 30, 40, 50 – richiede un numero elevato di campioni per suddividere un nodo, riducendo la varianza.
- `min_samples_leaf`: 15, 25, 35 – impone una dimensione minima per le foglie, ulteriore misura di regolarizzazione.
- `max_features`: `"sqrt"`, `"log2"`, 0.5 – controlla il numero di feature considerate a ogni split, aumentando la diversità tra gli alberi.

Inoltre, viene impostato `max_samples=0.75` per addestrare ogni albero su un sottocampione casuale (bootstrap) pari al 75% dei dati, riducendo ulteriormente la correlazione tra gli alberi.
La ricerca seleziona la combinazione che massimizza il C‑index medio in cross‑validazione.

---

### Fase 6: Valutazione Finale e Salvataggio degli Artefatti

Sul modello ottimizzato vengono calcolati:

- il C‑index sul training set (`ci_train`);
- il C‑index sul test set (`ci_test`).

I risultati vengono stampati per un rapido controllo di coerenza (il C‑index sul test non dovrebbe essere marcatamente inferiore a quello del training, indicando assenza di overfitting eccessivo).

Vengono salvati su disco:

- l’**imputer MICE** (`mice_imputer.pkl`) – necessario per riprodurre esattamente la stessa imputazione in fase di previsione su nuovi dati;
- il **modello RSF ottimizzato** (`rsf_model.pkl`) – pronto per la valutazione approfondita e per l’analisi explainable.

Il salvataggio separato dell’imputer e del modello consente di mantenere una pipeline modulare e facilmente riutilizzabile.

---

**Riepilogo delle principali modifiche rispetto alla versione precedente**:

- **Imputazione MICE** al posto di una semplice gestione dei NaN, per trattare i missing in modo multivariato.
- **Feature engineering** per introdurre variabili derivate clinicamente rilevanti.
- **Feature selection** automatica basata su permutation importance, per ridurre la dimensionalità e focalizzarsi sui predittori più informativi.
- **Griglia di iperparametri** più conservativa (alberi poco profondi, campioni minimi elevati) per contrastare l’overfitting.
- **Salvataggio separato di imputer e modello**, migliorando la riproducibilità e la facilità di deployment.

L’esecuzione di questa pipeline produce un modello Random Survival Forest ottimizzato, accompagnato da tutti gli oggetti necessari per le successive fasi di validazione e interpretazione.

In [5]:
import os, warnings
import numpy as np
import pandas as pd
import joblib
from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
DATA_PATH = "../data/dataset.csv"
MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, "rsf_model.pkl")
IMPUTER_PATH = os.path.join(MODEL_DIR, "mice_imputer.pkl")
os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL = "follow_ups.days_to_follow_up"
EVENT_COL = "demographic.vital_status_DEAD"
RANDOM_STATE = 25
TEST_SIZE = 0.25

# ==============================================================================
# FASE 1: CARICAMENTO E PREPROCESSING
# ==============================================================================
df = pd.read_csv(DATA_PATH)
df = df.replace(-1, np.nan)

exclude_cols = [TIME_COL, EVENT_COL] + [c for c in df.columns if df[c].dtype == 'O']
X = df.drop(columns=exclude_cols, errors="ignore")
y = Surv.from_arrays(event=df[EVENT_COL].astype(bool), time=df[TIME_COL])

# ==============================================================================
# FASE 2: SPLIT TRAIN / TEST
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[EVENT_COL]
)

# ==============================================================================
# FASE 3: IMPUTAZIONE MICE E FEATURE ENGINEERING
# ==============================================================================
imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=15)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

joblib.dump(imputer, IMPUTER_PATH)

def engineer_features(X_df):
    df_new = X_df.copy()
    if 'Final_T_enc' in df_new.columns and 'Final_N_enc' in df_new.columns:
        df_new['TNM_Burden'] = df_new['Final_T_enc'] + df_new['Final_N_enc']
        df_new['TNM_Interaction'] = df_new['Final_T_enc'] * df_new['Final_N_enc']

    trt_cols = [c for c in ['Trt_PHARMACEUTICAL', 'Trt_RADIATION', 'Trt_SURGERY'] if c in df_new.columns]
    if trt_cols:
        df_new['Total_Treatments'] = df_new[trt_cols].sum(axis=1)

    if 'demographic.age_at_index' in df_new.columns and 'TNM_Burden' in df_new.columns:
        df_new['Age_TNM_Risk'] = (df_new['demographic.age_at_index'] / 10) * df_new['TNM_Burden']

    return df_new

X_train_eng = engineer_features(X_train_imp)
X_test_eng = engineer_features(X_test_imp)

# ==============================================================================
# FASE 4: FEATURE SELECTION AUTOMATICA (TRAIN SET)
# ==============================================================================
selector = RandomSurvivalForest(n_estimators=100, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1)
selector.fit(X_train_eng, y_train)

result = permutation_importance(selector, X_train_eng, y_train, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
importances = result.importances_mean
top_k = min(30, X_train_eng.shape[1])
top_features = X_train_eng.columns[np.argsort(importances)[::-1][:top_k]]

X_train_sel = X_train_eng[top_features]
X_test_sel = X_test_eng[top_features]

# ==============================================================================
# FASE 5: OTTIMIZZAZIONE RSF E TRAINING
# ==============================================================================
param_distributions = {
    "n_estimators": [300, 500],
    "max_depth": [3, 4, 5],
    "min_samples_split": [30, 40, 50],
    "min_samples_leaf": [15, 25, 35],
    "max_features": ["sqrt", "log2", 0.5]
}

rsf_base = RandomSurvivalForest(random_state=RANDOM_STATE, n_jobs=-1, max_samples=0.75)

search = RandomizedSearchCV(
    rsf_base,
    param_distributions,
    n_iter=40,
    cv=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)
search.fit(X_train_sel, y_train)
best_model = search.best_estimator_

# ==============================================================================
# FASE 6: VALUTAZIONE E SALVATAGGIO
# ==============================================================================
ci_train = best_model.score(X_train_sel, y_train)
ci_test = best_model.score(X_test_sel, y_test)

joblib.dump(best_model, MODEL_PATH)

print("=" * 62)
print("  TRAINING COMPLETATO - RANDOM SURVIVAL FOREST")
print("=" * 62)
print(f"  CV C-index (best) : {search.best_score_:.4f}")
print(f"  C-index TRAIN     : {ci_train:.4f}")
print(f"  C-index TEST      : {ci_test:.4f}")
print(f"  Feature usate     : {list(top_features)}")
print(f"  Artefatti salvati in: {MODEL_DIR}/")
print("=" * 62)

  TRAINING COMPLETATO - RANDOM SURVIVAL FOREST
  CV C-index (best) : 0.6922
  C-index TRAIN     : 0.7444
  C-index TEST      : 0.7363
  Feature usate     : ['demographic.age_at_index', 'Intent_CURE', 'Age_TNM_Risk', 'diagnoses.laterality_RIGHT', 'exposures.alcohol_days_per_week', 'Final_N_enc', 'diagnoses.laterality_Missing', 'Total_Treatments', 'TNM_Interaction', 'cases.primary_site_FLOOR OF MOUTH', 'Trt_RADIATION', 'exposures.pack_years_smoked', 'Final_T_enc', 'exposures.alcohol_drinks_per_day', 'Trt_PHARMACEUTICAL', 'exposures.tobacco_smoking_status_enc', 'Trt_SURGERY', 'TNM_Burden', 'diagnoses.laterality_LEFT', 'cases.primary_site_LARYNX & HYPOPHARYNX', 'Intent_ADJUVANT', 'demographic.race_WHITE', 'gender_MALE', 'cases.primary_site_TONGUE', 'demographic.ethnicity_NOT HISPANIC OR LATINO', 'cases.primary_site_OROPHARYNX & TONSIL', 'demographic.race_NOT WHITE', 'cases.primary_site_LIP', 'Final_M_enc', 'demographic.ethnicity_HISPANIC OR LATINO']
  Artefatti salvati in: ../models/


# Random Survival Forest (RSF) – Evaluation & Explainable AI (Versione Aggiornata)

Questo notebook valuta le performance e le logiche decisionali del modello **Random Survival Forest** addestrato con la pipeline aggiornata.
Rispetto alla versione precedente, l’analisi è stata arricchita con metriche aggiuntive (cross‑validazione, IBS, AUC tempo‑dipendente) e una maggiore attenzione alla riproducibilità del preprocessing (imputazione, feature engineering, selezione delle feature) in fase di test.

---

### Fase 1: Ricostruzione dell’Ambiente e Preprocessing

Per garantire che la valutazione avvenga nelle stesse condizioni del training, l’intera pipeline di preprocessing viene riprodotta fedelmente:

1. **Caricamento e pulizia** – Il dataset viene letto e i placeholder `-1` convertiti in `NaN`, esattamente come in fase di addestramento.
2. **Suddivisione train/test** – Utilizzando lo stesso `random_state` e la stessa frazione di test, si isolano gli stessi insiemi usati durante il training, garantendo che il modello venga valutato sui dati mai visti durante la fase di ottimizzazione.
3. **Imputazione MICE** – Viene caricato l’imputer salvato (`mice_imputer.pkl`) e applicato separatamente a training e test. Questo passaggio è fondamentale perché l’imputer è stato “addestrato” solo sul training set e la sua applicazione al test set evita qualsiasi forma di *data leakage*.
4. **Feature engineering** – Sulle matrici imputate vengono create le stesse feature derivate introdotte nel training:
   - `TNM_Burden` = `Final_T_enc` + `Final_N_enc`
   - `TNM_Interaction` = `Final_T_enc` * `Final_N_enc`
   - `Total_Treatments` (somma dei trattamenti)
   - `Age_TNM_Risk` = (età / 10) * `TNM_Burden`
5. **Selezione delle feature** – Il modello RSF ottimizzato memorizza l’elenco delle feature effettivamente usate (`feature_names_in_`). I dataframe di test vengono quindi filtrati per contenere solo quelle colonne, nell’ordine atteso dal modello.

Questa ricostruzione meticolosa assicura che le predizioni e le metriche calcolate riflettano fedelmente le performance reali del modello su nuovi dati.

---

### Fase 2: Metriche Globali e Performance

Viene condotta un’analisi quantitativa completa, integrando diverse metriche di sopravvivenza:

- **Cross‑validazione (5‑fold) sul training set** – Per ottenere una stima robusta della performance attesa del modello, si esegue una cross‑validazione a 5 fold utilizzando i soli dati di training. Il punteggio medio fornisce un’indicazione della stabilità del modello e della sua capacità di generalizzare.
- **C‑index su training e test** – Calcolato con il metodo `score` del modello, misura la capacità di discriminazione (ranking corretto dei rischi).
- **Integrated Brier Score (IBS)** – Valuta la calibrazione delle probabilità di sopravvivenza lungo l’intero arco temporale. Le funzioni di sopravvivenza predette dal modello vengono valutate su una griglia di 20 punti compresi tra il 10° e il 90° percentile dei tempi di test. I valori sono ritagliati per evitare instabilità numeriche. L’IBS è una metrica particolarmente informativa perché premia modelli ben calibrati oltre che discriminativi.
- **AUC tempo‑dipendente** – Utilizzando `cumulative_dynamic_auc` di `scikit‑survival`, si calcola l’area sotto la curva a orizzonti clinicamente rilevanti: 1 anno (365 giorni), 3 anni (1095 giorni) e 5 anni (1825 giorni). Vengono considerati solo gli orizzonti che ricadono entro il range di follow‑up effettivo del test set. I risultati includono l’AUC per ogni orizzonte valido e la loro media, offrendo una visione della capacità discriminativa del modello in specifiche finestre temporali.

Questo insieme di metriche fornisce un quadro completo delle performance del modello, andando oltre il semplice C‑index e includendo aspetti di calibrazione e di evoluzione temporale del potere predittivo.

---

### Fase 3: Stratificazione del Rischio (Validazione Clinica)

I pazienti del test set vengono divisi in due gruppi in base alla **mediana** del risk score predetto (output del modello RSF):

- **Alto Rischio** – risk score superiore alla mediana
- **Basso Rischio** – risk score ≤ mediana

Per ciascun gruppo si calcola la **curva di sopravvivenza media**, ottenuta valutando le funzioni di sopravvivenza individuali su una griglia temporale comune e mediando i valori. Il grafico a gradini risultante mostra chiaramente la separazione tra le due coorti: una buona stratificazione si traduce in curve distinte e ben separate, con i pazienti ad alto rischio che presentano una probabilità di sopravvivenza sistematicamente inferiore.

Questa visualizzazione ha un forte valore clinico, in quanto permette di apprezzare immediatamente l’utilità pratica del modello nel distinguere tra pazienti con prognosi diversa.

---

### Fase 4: Explainability Globale – Permutation Feature Importance

Per identificare quali variabili guidano maggiormente le predizioni a livello di popolazione, si applica la **Permutation Feature Importance**:

- Il C‑index viene ricalcolato dopo aver permutato casualmente i valori di ciascuna feature (10 ripetizioni). La riduzione media del C‑index rispetto al valore originale rappresenta l’importanza della feature.
- Le importanze vengono convertite in **percentuali relative** (normalizzate a somma 100%) per facilitare l’interpretazione.
- Il grafico a barre orizzontali (con deviazione standard) evidenzia i principali driver predittivi, confermando o arricchendo le conoscenze cliniche.

Questo metodo è indipendente dal modello e fornisce una visione globale dell’impatto di ogni variabile sulle prestazioni, senza assumere linearità o altre proprietà specifiche.

---

### Fase 5: Explainability Locale – SHAP (SHapley Additive exPlanations)

Per comprendere come ciascuna feature contribuisce al rischio di un **singolo paziente**, si utilizza SHAP. La procedura ricalca quella adottata per XGBSE:

- Viene estratto un **background dataset** di 100 campioni dal training set (campionamento casuale con seed fisso) per inizializzare l’`Explainer`. L’explainer è costruito sulla funzione `rsf.predict` (che restituisce direttamente i risk scores).
- **SHAP Summary Plot (beeswarm)** – Mostra, per ogni feature, la distribuzione dei valori SHAP sull’intero test set. Il colore indica il valore della feature (rosso = alto, blu = basso), rivelando l’effetto monotono e l’importanza globale.
- **SHAP Waterfall Plot** – Generato per i due pazienti con il rischio **più alto** e **più basso** nel test set. Questi grafici partono dal valore base (il rischio medio atteso sulla popolazione di background) e aggiungono/sottraggono i contributi SHAP di ogni feature, illustrando in modo trasparente il ragionamento matematico che porta al rischio finale del singolo paziente.

Tutti i grafici vengono salvati nella cartella `results/` per una successiva inclusione in report o presentazioni.

---

**Riepilogo delle novità rispetto alla versione precedente**:

- **Cross‑validazione** sul training set per una stima più robusta delle performance.
- **Integrated Brier Score (IBS)** per valutare la calibrazione delle curve di sopravvivenza.
- **AUC tempo‑dipendente** a orizzonti clinici (1, 3, 5 anni) per analizzare l’evoluzione del potere discriminativo.
- **Ricostruzione fedele della pipeline di preprocessing** (imputazione, feature engineering, selezione feature) in fase di test, garantendo coerenza con il training.
- **SHAP** con background sampling e generazione automatica di waterfall per i pazienti estremi.

L’esecuzione di questo notebook fornisce un quadro completo delle performance e delle logiche del modello Random Survival Forest, soddisfacendo sia le esigenze di validazione quantitativa che quelle di interpretabilità clinica.

In [6]:
import os, warnings
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance
from sksurv.util import Surv
from sksurv.metrics import (integrated_brier_score,
                            cumulative_dynamic_auc)

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
RESULTS_DIR = "results"
DATA_PATH = "../data/dataset.csv"
MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, "rsf_model.pkl")
IMPUTER_PATH = os.path.join(MODEL_DIR, "mice_imputer.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

for _f, _name in [(MODEL_PATH, "rsf_model.pkl"), (IMPUTER_PATH, "mice_imputer.pkl")]:
    if not os.path.exists(_f):
        raise FileNotFoundError(f"{_name} non trovato in {MODEL_DIR}.\nEsegui prima il training.")

TIME_COL = "follow_ups.days_to_follow_up"
EVENT_COL = "demographic.vital_status_DEAD"
RANDOM_STATE = 25
TEST_SIZE = 0.25

# ==============================================================================
# 1. CARICAMENTO DATI, IMPUTAZIONE E MODELLO
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - RANDOM SURVIVAL FOREST (RSF)")
print("=" * 62)

df = pd.read_csv(DATA_PATH)
df = df.replace(-1, np.nan)

X_all = df.drop(columns=[TIME_COL, EVENT_COL])
t_all = df[TIME_COL].values
e_all = df[EVENT_COL].values.astype(int)

X_train_raw, X_test_raw, t_train, t_test, e_train, e_test = train_test_split(
    X_all, t_all, e_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=e_all,
)

imputer = joblib.load(IMPUTER_PATH)
rsf = joblib.load(MODEL_PATH)

X_train_imp = pd.DataFrame(imputer.transform(X_train_raw), columns=X_train_raw.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_raw), columns=X_test_raw.columns)


def engineer_features(X_df):
    df_new = X_df.copy()
    if 'Final_T_enc' in df_new.columns and 'Final_N_enc' in df_new.columns:
        df_new['TNM_Burden'] = df_new['Final_T_enc'] + df_new['Final_N_enc']
        df_new['TNM_Interaction'] = df_new['Final_T_enc'] * df_new['Final_N_enc']

    trt_cols = [c for c in ['Trt_PHARMACEUTICAL', 'Trt_RADIATION', 'Trt_SURGERY'] if c in df_new.columns]
    if trt_cols:
        df_new['Total_Treatments'] = df_new[trt_cols].sum(axis=1)

    if 'demographic.age_at_index' in df_new.columns and 'TNM_Burden' in df_new.columns:
        df_new['Age_TNM_Risk'] = (df_new['demographic.age_at_index'] / 10) * df_new['TNM_Burden']

    return df_new


X_train_eng = engineer_features(X_train_imp)
X_test_eng = engineer_features(X_test_imp)

expected_features = rsf.feature_names_in_
X_train = X_train_eng[expected_features]
X_test = X_test_eng[expected_features]

y_train = Surv.from_arrays(event=e_train.astype(bool), time=t_train)
y_test = Surv.from_arrays(event=e_test.astype(bool), time=t_test)

risk_scores = rsf.predict(X_test)
print(f"[+] Modello caricato | Test set: {len(X_test)} pz | {e_test.sum()} eventi\n")

# ==============================================================================
# FASE 2: METRICHE GLOBALI E CROSS-VALIDATION
# ==============================================================================
print("=" * 62)
print("  FASE 2: METRICHE GLOBALI E PERFORMANCE")
print("=" * 62)

print("  Calcolo Cross-Validation (5-fold) in corso...")
cv_scores = cross_val_score(rsf, X_train, y_train, cv=5, n_jobs=-1)
cv_mean_score = cv_scores.mean()

c_index_train = rsf.score(X_train, y_train)
c_index_test = rsf.score(X_test, y_test)

lb = np.percentile(t_test, 10)
ub = np.percentile(t_test, 90)
ibs_times = np.linspace(lb, ub, 20)

surv_funcs = rsf.predict_survival_function(X_test)
preds_ibs = np.clip(np.vstack([fn(ibs_times) for fn in surv_funcs]), 1e-6, 1 - 1e-6)
ibs = integrated_brier_score(y_train, y_test, preds_ibs, ibs_times)

tempi_giorni = [365, 1095, 1825]
etichette = ["1 anno (365 gg)", "3 anni (1095 gg)", "5 anni (1825 gg)"]
min_t, max_t = t_test.min(), t_test.max()

tempi_validi, label_validi = [], []
for t_g, lbl in zip(tempi_giorni, etichette):
    if min_t < t_g < max_t:
        tempi_validi.append(t_g)
        label_validi.append(lbl)

auc_vals, mean_auc = None, None
if tempi_validi:
    auc_vals, mean_auc = cumulative_dynamic_auc(y_train, y_test, risk_scores, np.array(tempi_validi))

print(f"  CV C-index (5-fold): {cv_mean_score:.4f}")
print(f"  C-index TRAIN      : {c_index_train:.4f}")
print(f"  C-index TEST       : {c_index_test:.4f}")
print(f"  IBS (Test)         : {ibs:.4f} (baseline ~0.25, ↓ meglio)")
if mean_auc:
    print(f"  Mean AUC (Test)    : {mean_auc:.4f}")
    for lbl, auc in zip(label_validi, auc_vals):
        print(f"    - AUC a {lbl} : {auc:.4f}")

# ==============================================================================
# FASE 3: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)")
print("=" * 62)

median_risk = np.median(risk_scores)
mask_high = risk_scores > median_risk
mask_low = ~mask_high
plot_times = np.linspace(lb, ub, 100)

surv_high = rsf.predict_survival_function(X_test[mask_high])
surv_low = rsf.predict_survival_function(X_test[mask_low])
mean_surv_high = np.mean(np.vstack([fn(plot_times) for fn in surv_high]), axis=0)
mean_surv_low = np.mean(np.vstack([fn(plot_times) for fn in surv_low]), axis=0)

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.step(plot_times, mean_surv_low, where="post", label="Basso Rischio", color="#2ca02c", lw=2.5)
ax1.step(plot_times, mean_surv_high, where="post", label="Alto Rischio", color="#d62728", lw=2.5)
ax1.fill_between(plot_times, mean_surv_low, mean_surv_high, alpha=0.08, color="grey", step="post")
ax1.set_title("Curve di Sopravvivenza (RSF): Alto vs Basso Rischio", fontsize=13, fontweight="bold")
ax1.set_ylabel("Probabilità di Sopravvivenza", fontsize=11)
ax1.set_xlabel("Giorni di Follow-up", fontsize=11)
ax1.set_ylim(0, 1.02)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.35, linestyle="--")
plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "rsf_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 4: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")

perm = permutation_importance(
    rsf, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature": X_test.columns,
    "Importanza (%)": imp_pct,
    "Std": perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

df_imp_top = df_imp.head(15)

fig2, ax2 = plt.subplots(figsize=(9, 7))
ax2.barh(df_imp_top["Feature"][::-1], df_imp_top["Importanza (%)"][::-1], color="#1f77b4", xerr=df_imp_top["Std"][::-1],
         capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – RSF", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "rsf_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 5: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 5: XAI LOCALE (SHAP)")
print("=" * 62)


def _predict_risk_shap(X_raw):
    if isinstance(X_raw, pd.DataFrame):
        return rsf.predict(X_raw)
    return rsf.predict(pd.DataFrame(X_raw, columns=X_test.columns))


background = shap.sample(X_train, 100, random_state=RANDOM_STATE)
explainer_shap = shap.Explainer(_predict_risk_shap, background)

print("  Calcolo SHAP su tutto il test set...")
shap_values_all = explainer_shap(X_test)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values_all, max_display=15, show=False)
plt.title("SHAP Summary – RSF (Contributo al Risk Score)", fontsize=12, fontweight="bold")
plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "rsf_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto = int(np.argmax(risk_scores))
idx_basso = int(np.argmin(risk_scores))

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False, matplotlib=True)
    except TypeError:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – RSF – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"rsf_shap_waterfall_{label.lower().replace(' ', '_')}.png"

    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – Random Survival Forest")
print("=" * 62)
print(f" Esecuzione completata con successo. Tutti i grafici sono salvati in:\n  {RESULTS_DIR}")

  EVALUATION & XAI - RANDOM SURVIVAL FOREST (RSF)
[+] Modello caricato | Test set: 132 pz | 56 eventi

  FASE 2: METRICHE GLOBALI E PERFORMANCE
  Calcolo Cross-Validation (5-fold) in corso...
  CV C-index (5-fold): 0.6922
  C-index TRAIN      : 0.7444
  C-index TEST       : 0.7363
  IBS (Test)         : 0.1923 (baseline ~0.25, ↓ meglio)
  Mean AUC (Test)    : 0.7844
    - AUC a 1 anno (365 gg) : 0.7757
    - AUC a 3 anni (1095 gg) : 0.7881
    - AUC a 5 anni (1825 gg) : 0.7896

  FASE 3: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)
  [+] Grafico curve salvato in: results

  FASE 4: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: results

  FASE 5: XAI LOCALE (SHAP)
  Calcolo SHAP su tutto il test set...


PermutationExplainer explainer: 133it [17:16,  7.91s/it]                         


  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio

  RIEPILOGO FINALE – Random Survival Forest
 Esecuzione completata con successo. Tutti i grafici sono salvati in:
  results
